## Le pipeline qu'on va construire

```
  📄 Documents PDF
        ↓
  ✂️  Découpage en chunks
        ↓
  🧠 Transformation en embeddings (vecteurs)
        ↓
  🗄️  Stockage dans une base vectorielle
        ↓
  ❓ Question utilisateur
        ↓
  🔎 Recherche des chunks pertinents
        ↓
  🤖 LLM génère une réponse basée sur les chunks trouvés
```

Chaque étape est une cellule du notebook.

---
## 🔧 Étape 0 — Installation et configuration

On installe les librairies nécessaires :
- **`openai`** : pour interroger GPT (le LLM)
- **`sentence-transformers`** : pour créer les embeddings (fonctionne en local, gratuit)
- **`chromadb`** : base de données vectorielle légère
- **`pypdf`** : lecture des PDF

In [ ]:
!pip install -q openai sentence-transformers chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

### 🔑 Configuration de la clé API OpenAI

Votre enseignant vous a fourni une clé API. **Ne la collez JAMAIS directement dans le code** (sinon elle fuite si vous partagez le notebook).

**Méthode propre avec Colab Secrets** :
1. Cliquez sur l'icône 🔑 (clé) dans la barre latérale gauche de Colab
2. Ajoutez un nouveau secret nommé `OPENAI_API_KEY`
3. Collez la valeur de la clé
4. Activez le toggle "Accès au notebook"
5. Exécutez la cellule ci-dessous

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("Clé API configurée")

Clé API configurée


---
## Étape 1 — Charger un document PDF

On va lire un PDF et extraire son texte.

**Action** : uploadez un PDF dans Colab (bouton 📁 à gauche → "Importer"), puis modifiez le nom du fichier dans la cellule ci-dessous.

In [ ]:
from pypdf import PdfReader

# Modifiez le nom selon votre fichier
CHEMIN_PDF = "Livre-blanc-ecologie-pour-quoi-faire.pdf"

reader = PdfReader(CHEMIN_PDF)

texte_complet = ""
for page in reader.pages:
    texte_complet += page.extract_text() + "\n"

print(f"📄 Document chargé : {len(texte_complet)} caractères")
print(f"📑 Nombre de pages : {len(reader.pages)}")
print("\n--- Aperçu des 500 premiers caractères ---")
print(texte_complet[:500])

📄 Document chargé : 455363 caractères
📑 Nombre de pages : 101

--- Aperçu des 500 premiers caractères ---
L’ÉCOLOGIE,  
POUR QUOI FAIRE ?
70 propositions  
pour la santé, le pouvoir d'achat,  
le vivre-ensemble, la souveraineté, 
la sécurité, la démocratie... 

L’ÉCOLOGIE,  
POUR QUOI FAIRE ?
70 propositions  
pour la santé, le pouvoir d'achat,  
le vivre-ensemble, la souveraineté, 
la sécurité, la démocratie... 
4
L’écologie, pour quoi faire ?  .......................................................................................................................... 14 
Par Estelle Brachlianoff, Dir


---
## Étape 2 — Découper en chunks

On va découper le texte en **morceaux** (chunks) de taille gérable.

**Paramètres importants :**
- `taille` : nombre de caractères par chunk
- `chevauchement` : caractères partagés entre deux chunks consécutifs (pour éviter de couper une phrase en plein milieu)

In [ ]:
def decouper_en_chunks(texte, taille=2000, chevauchement=200):
    """Découpe un texte en chunks avec chevauchement."""
    chunks = []
    debut = 0
    while debut < len(texte):
        fin = debut + taille
        chunks.append(texte[debut:fin])
        debut += taille - chevauchement  # on recule de `chevauchement` caractères
    return chunks

chunks = decouper_en_chunks(texte_complet, taille=2000, chevauchement=200)

print(f"{len(chunks)} chunks créés")
print(f"\n--- Premier chunk ---\n{chunks[0]}")
print(f"\n--- Deuxième chunk (avec chevauchement) ---\n{chunks[1]}")

253 chunks créés

--- Premier chunk ---
L’ÉCOLOGIE,  
POUR QUOI FAIRE ?
70 propositions  
pour la santé, le pouvoir d'achat,  
le vivre-ensemble, la souveraineté, 
la sécurité, la démocratie... 

L’ÉCOLOGIE,  
POUR QUOI FAIRE ?
70 propositions  
pour la santé, le pouvoir d'achat,  
le vivre-ensemble, la souveraineté, 
la sécurité, la démocratie... 
4
L’écologie, pour quoi faire ?  .......................................................................................................................... 14 
Par Estelle Brachlianoff, Directrice générale de Veolia
L'écologie concrète,  
au cœur des préoccupations des Français   ...................................................... 16 
Par Jean-François Nogrette, Directeur zone France & DSE de Veolia
Les entreprises, des partenaires clés   ...............................................................................18
Innover pour décarboner, 
dépolluer et régénérer les ressources  
pour les services essentiels   ........................

---
## Étape 3 — Créer les embeddings

Un **embedding**, c'est une représentation d'un texte sous forme de **vecteur de nombres**.

Deux textes proches en **sens** auront des vecteurs proches, même s'ils n'ont aucun mot en commun.

In [ ]:
from sentence_transformers import SentenceTransformer

print("⏳ Chargement du modèle d'embeddings (peut prendre 30s la première fois)...")
model_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Modèle chargé")

# On encode tous les chunks
embeddings = model_embeddings.encode(chunks, show_progress_bar=True)

print(f"\nShape des embeddings : {embeddings.shape}")
print(f"   → {embeddings.shape[0]} chunks, chacun représenté par un vecteur de {embeddings.shape[1]} dimensions")
print(f"\nExemple : les 10 premières valeurs du premier embedding :")
print(embeddings[0][:10])

⏳ Chargement du modèle d'embeddings (peut prendre 30s la première fois)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modèle chargé


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Shape des embeddings : (253, 384)
   → 253 chunks, chacun représenté par un vecteur de 384 dimensions

Exemple : les 10 premières valeurs du premier embedding :
[ 0.0615424  -0.00134769  0.11422631 -0.00160315  0.2525903   0.18335862
 -0.03240896  0.1923556  -0.11630399  0.16758956]


### Démo : les embeddings capturent le sens

Testez la similarité entre différents mots. Deux textes proches en sens ont une similarité proche de 1, deux textes sans rapport proche de 0.

In [ ]:
from sentence_transformers.util import cos_sim

paires = [
    ("sa va", "ca va"),
]

for texte_a, texte_b in paires:
    emb_a = model_embeddings.encode(texte_a)
    emb_b = model_embeddings.encode(texte_b)
    similarite = cos_sim(emb_a, emb_b).item()
    print(f"{similarite:.3f}  |  '{texte_a}' ↔ '{texte_b}'")

0.906  |  'sa va' ↔ 'ca va'


**Observez** : "chat" et "félin" sont très proches (~0.7) alors qu'ils n'ont **aucune lettre en commun**. C'est exactement ce qu'on veut pour chercher par sens plutôt que par mots-clés.

*Pour visualiser les embeddings en 3D, testez aussi : [projector.tensorflow.org](https://projector.tensorflow.org/)*

---
## Étape 4 — Stocker dans une base vectorielle

Une **base vectorielle** est optimisée pour trouver rapidement les vecteurs les plus proches d'un vecteur donné (la question).

On utilise **ChromaDB** : simple, gratuit, tourne en mémoire.

In [ ]:
import chromadb

# Créer la base (en mémoire, donc éphémère - reset à chaque redémarrage)
client = chromadb.Client()

# Supprimer si elle existe déjà (au cas où on relance la cellule)
try:
    client.delete_collection("mes_docs")
except:
    pass

collection = client.create_collection("mes_docs")

# Ajouter les chunks + leurs embeddings
collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"{collection.count()} chunks stockés dans la base vectorielle")

253 chunks stockés dans la base vectorielle


---
## Étape 5 — Rechercher les chunks pertinents

Maintenant, quand on pose une question, on :
1. Transforme la question en embedding
2. Cherche les chunks les plus similaires dans la base$

In [ ]:
# Adaptez la question au contenu de VOTRE PDF
question = "qui a écrit livre blanc ecologie ?"

# Transformer la question en embedding
embedding_question = model_embeddings.encode([question])

# Chercher les 3 chunks les plus pertinents
resultats = collection.query(
    query_embeddings=embedding_question.tolist(),
    n_results=3
)

print(f"❓ Question : {question}\n")
print(f"Top 3 chunks les plus pertinents :\n")

for i, (chunk, distance) in enumerate(zip(resultats['documents'][0], resultats['distances'][0])):
    print(f"--- Chunk #{i+1} (distance: {distance:.3f}) ---")
    print(chunk)
    print()

❓ Question : qui a écrit livre blanc ecologie ?

Top 3 chunks les plus pertinents :

--- Chunk #1 (distance: 20.626) ---
ansformation écologique non 
pas une contrainte, mais une formidable opportunité de progrès et 
d’innovation au service des générations présentes et futures. C’est 
avec cette conviction et cet engagement que je vous invite à décou-
vrir et à vous approprier ces propositions.
Estelle Brachlianoff,
Directrice générale de Veolia
15
L ’écologie : pour quoi faire ? EDITO
N
ous sommes à la croisée des chemins. Les urgences économiques, sociales et 
sanitaires pourraient conduire certains à réévaluer les priorités, au risque de 
marginaliser les questions environnementales. Pourtant, notre deuxième édi-
tion du « Baromètre de la Transformation Écologique » révèle une réalité diffé-
rente : l'opinion publique française reste résolument tournée vers l'avenir, consciente des 
enjeux environnementaux et prête à s'engager. Face au dérèglement climatique, l'heure 
n'est en rien 

Note importante

La **distance** indique à quel point le chunk est "proche" de la question. Plus c'est faible, plus c'est pertinent.

Le RAG trouve les bons passages **sans recherche par mots-clés**. Même si votre question utilise des synonymes, il trouvera quand même.

---
## Étape 6 — Générer la réponse avec le LLM

On donne au LLM :
1. Les chunks trouvés (le **contexte**)
2. La question de l'utilisateur
3. Des instructions claires sur comment répondre

In [ ]:
from openai import OpenAI

client_openai = OpenAI()

def generer_reponse(question, chunks_pertinents):
    # Assembler le contexte
    contexte = "\n\n---\n\n".join(chunks_pertinents)

    # Construire le prompt
    prompt = f"""Tu es un assistant expert qui répond à des questions en te basant UNIQUEMENT sur le contexte fourni.

Règles :
- Si la réponse est dans le contexte, réponds précisément et cite le passage
- Si la réponse n'est PAS dans le contexte, dis explicitement "Cette information n'est pas dans les documents fournis"
- Ne fais JAMAIS d'hypothèses basées sur tes connaissances générales

CONTEXTE :
{contexte}

QUESTION : {question}

RÉPONSE :"""

    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",  # rapide et pas cher
        messages=[{"role": "user", "content": prompt}],
        temperature=0  # 0 = déterministe, réponses factuelles
    )

    return response.choices[0].message.content

# Générer la réponse
reponse = generer_reponse(question, resultats['documents'][0])
print(f"❓ Question : {question}\n")
print(f"Réponse :\n{reponse}")

❓ Question : qui a écrit livre blanc ecologie ?

Réponse :
Cette information n'est pas dans les documents fournis.


---
## Étape 7 — Tout regrouper dans une fonction `rag()`

Maintenant on met tout le pipeline dans une seule fonction réutilisable.

In [ ]:
def rag(question, n_chunks=3, verbose=False):
    """Pipeline RAG complet : de la question à la réponse."""

    # 1. Embedding de la question
    emb_q = model_embeddings.encode([question])

    # 2. Recherche des chunks pertinents
    resultats = collection.query(
        query_embeddings=emb_q.tolist(),
        n_results=n_chunks
    )

    chunks_trouves = resultats['documents'][0]

    if verbose:
        print("🔎 Chunks trouvés :")
        for i, c in enumerate(chunks_trouves):
            print(f"  [{i+1}] {c[:100]}...")
        print()

    # 3. Génération de la réponse
    return generer_reponse(question, chunks_trouves)


# Test avec une question
reponse = rag("Quelle est l'idée principale du document ?", verbose=True)
print(f"\n Réponse finale :\n{reponse}")

🔎 Chunks trouvés :
  [1] rformances  
en le challengeant à intervalles définis
Pouvoir se séparer de son prestataire est un m...
  [2]  Academia comptera 1 campus 
par région française et dès 2027, elle déploiera son offre à 
l’interna...
  [3] formance éner -
gétique : identifier des méthodologies duplicables sur les 
territoires pour gagner ...


 Réponse finale :
Cette information n'est pas dans les documents fournis.


---
## Étape 8 — À vous de jouer

### Exercice 1 — Posez 5 questions à votre document

Choisissez 5 questions variées :
- Une factuelle (date, chiffre, nom…)
- Une qui demande un résumé
- Une qui demande une comparaison
- Une sur un détail précis
- Une **dont la réponse N'EST PAS dans le document** (pour tester si le RAG ne hallucine pas)

In [ ]:
# À vous ! Remplacez les questions par les vôtres

mes_questions = [
    "???",
    "???",
    "???",
    "???",
    "???",
]

for q in mes_questions:
    print(f" {q}")
    print(f" {rag(q)}")
    print("-" * 60)

### Exercice 2 — Comparer avec / sans RAG

Posez la même question directement au LLM (sans contexte) et avec le RAG. Vous verrez la différence : sans RAG, le LLM invente ou dit qu'il ne sait pas.

In [ ]:
def llm_sans_rag(question):
    """Interroger GPT sans aucun contexte."""
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0
    )
    return response.choices[0].message.content


question_test = "que propose Veolia pour l'éoclogie ?"  # mettez une question spécifique à VOTRE document

print("SANS RAG (LLM seul) :")
print(llm_sans_rag(question_test))
print("\n" + "="*60 + "\n")
print("AVEC RAG :")
print(rag(question_test))

SANS RAG (LLM seul) :
Veolia est une entreprise française spécialisée dans la gestion de l'eau, des déchets et des services énergétiques. Elle propose plusieurs initiatives et solutions pour promouvoir l'écologie et le développement durable. Voici quelques-unes de ses propositions :

1. **Gestion de l'eau** : Veolia travaille à l'optimisation de la gestion des ressources en eau, en proposant des solutions pour le traitement des eaux usées, la réutilisation de l'eau et la réduction des fuites dans les réseaux de distribution.

2. **Recyclage et valorisation des déchets** : L'entreprise met en place des systèmes de collecte et de tri des déchets pour favoriser le recyclage. Elle développe également des technologies pour valoriser les déchets, par exemple en produisant de l'énergie à partir de déchets organiques.

3. **Économie circulaire** : Veolia s'engage à promouvoir l'économie circulaire en transformant les déchets en ressources. Cela inclut la récupération de matériaux recyclables e

### Exercice 3 — Jouer sur les paramètres

Essayez de modifier :
- `n_chunks` dans la fonction `rag()` : 1, 3, 5, 10... → effet sur la qualité ?
- `taille` du chunking au début (`decouper_en_chunks`) : 200, 500, 1500... → effet ?
- La `temperature` du LLM : 0, 0.7, 1.5... → effet sur la créativité de la réponse ?

** Pour changer le chunking, vous devrez relancer toutes les cellules depuis l'étape 2.**